# The per-window thresholds Λ_w (Appendix A.2)

**In the paper:** App. A.2, "the calibrated thresholds are 11.1 for adding a second source and 10.8 for adding a third".

The per-window ladder (Fig. 1) adds a source to a window with $K$ sources when

$$D_K \equiv \frac{\chi^2_K - \chi^2_{K+1}}{\sigma_n^2} > \Delta_K, \qquad \Lambda_{\rm w} = (\Delta_1, \Delta_2),$$

and $\Delta_K$ is the 99th percentile of $D_K$ over simulated windows that hold exactly $K$ sources, each reduced by
the ladder itself: a spurious source is added in 1 % of the windows the $K$-source model already explains.

| step | task | calls | paper |
|---|---|---|---|
| 1 | the null windows of each rung: single stars (rung 1), pairs at 0.5–2.5 FWHM and flux ratios 0.1–1 (rung 2); 20 000 each | `sim.snr_from_peak_snr`, `sim.rung_null_statistic` | App. A.2 |
| 2 | one null window reduced by hand: its $D_1$ | `sim.simulate_window`, `red.fit_k` | App. A.2 |
| 3 | the stored null is the package's: the first 200 windows of each rung recomputed | `sim.rung_null_statistic` | — |
| 4 | the margins: the 99th percentiles, equal to the frozen `ins.TOY_MARGINS`; why not a χ² table | — (the constants `sim.MARGIN_PERCENTILE`, `ins.TOY_MARGINS`) | App. A.2 |
| 5 | the null distributions | — | — |

Every scanlib call carries the name of its module (`dyn.wls`, `joint.fit_joint`); the first code cell lists the
modules.

**What it reproduces.** $\Delta_1 = 11.1$, $\Delta_2 = 10.8$, equal to `ins.TOY_MARGINS` to the last bit.
**Run time.** About 15 s; `FULL = True` recomputes the 40 000 windows (20–50 min) and rewrites `../stored/margin_nulls.npz`.

In [ ]:
# the imports: numpy, matplotlib, scipy's chi^2, and the scanlib modules; every scanlib call below carries its
# module's name
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import chi2

from scanlib import instrument as ins      # the instrument: sampling, LSF, the along-scan projection (Eq. 9)
from scanlib import simulation as sim      # synthetic scenes and their windows
from scanlib import reduction as red       # one window, or a campaign, reduced to detections

In [ ]:
# the instrument, defined once (Sect. 2.1): 18 samples of 59 mas along scan, a Gaussian LSF of FWHM 100 mas,
# and its per-window margins Delta_1, Delta_2 (App. A.2; derived in sim_margins_lambda_w)
FWHM_MAS, PITCH_MAS, N_SAMPLES = 100.0, 59.0, 18
inst = ins.Instrument.gaussian(fwhm_mas=FWHM_MAS, pitch_mas=PITCH_MAS, n_samples=N_SAMPLES, margins=ins.TOY_MARGINS)

In [ ]:
# the switch FULL, and the store of the null statistics
FULL = False                               # True: recompute the 40 000 null windows (20-50 min)
STORE = Path("..") / "stored" / "margin_nulls.npz"

## 1. The null windows (App. A.2)

Each rung draws from its own random stream, spawned from one seed as
`calibrate_margins` spawns them, so a rung can be recomputed on its own.
The null windows are rendered at the regime the margins were frozen in, peak
S/N 50 of the earlier convention (S/N 115 today, converted exactly by
`sim.snr_from_peak_snr`). The statistic is residual power over $\sigma_n^2$ of
an adequate model, so it does not depend on the regime while the LSF is right.

In [ ]:
# the rungs, the regime of the null windows, and the random streams, one per rung
RUNGS = (1, 2)                             # rung K tests a window of K sources for one more source
NULL_PEAK_SNR = 50.0                       # the S/N of the null windows in the earlier convention (peak S/N)
# the regime of the frozen margins, in today's S/N convention
NULL_SNR = sim.snr_from_peak_snr(NULL_PEAK_SNR, inst=inst)


def streams():
    """The random generators of the null windows, one per rung.

    Returns a list of two generators: [0] is rung 1's, [1] is rung 2's. One generator is made from the seed
    ins.TOY_MARGINS_SEED, and spawn(2) derives two independent child generators from it, as calibrate_margins
    spawns them, so each rung can be redrawn alone. Every call returns the two generators at their start."""
    return np.random.default_rng(ins.TOY_MARGINS_SEED).spawn(2)

In [ ]:
# the null statistic D_K of each rung: recomputed and stored with FULL, otherwise loaded from the store
# nulls: a table from a rung's number (1 or 2) to the D_K of each of its null windows; nulls[1] is rung 1's
if FULL:
    # [r - 1]: rung r has the index r - 1 in the streams and in ins.TOY_MARGINS_N
    rung_streams = streams()
    # sim.rung_null_statistic: D_K of n simulated windows that hold exactly K = r sources, each reduced by the ladder
    nulls = {}
    for r in RUNGS:                        # r: the rung's number
        rng_of_rung = rung_streams[r - 1]
        n_windows = ins.TOY_MARGINS_N[r - 1]               # the number of null windows of the rung
        nulls[r] = sim.rung_null_statistic(rng_of_rung, r, n_windows, snr=NULL_SNR, inst=inst)
    STORE.parent.mkdir(exist_ok=True)
    # the store: the two nulls, the seed, the number of windows of each rung, and the regime
    n_per_rung = []
    for r in RUNGS:
        n_per_rung.append(ins.TOY_MARGINS_N[r - 1])
    np.savez(STORE, rung1=nulls[1], rung2=nulls[2], seed=ins.TOY_MARGINS_SEED,
             n_per_rung=np.array(n_per_rung), peak_snr=NULL_PEAK_SNR)
    print(f"computed and stored {STORE}")
else:
    with np.load(STORE) as stored:
        # the store was made with the package's seed and at this notebook's regime
        assert int(stored["seed"]) == ins.TOY_MARGINS_SEED and float(stored["peak_snr"]) == NULL_PEAK_SNR
        nulls = {}
        for r in RUNGS:                    # r: the rung's number; the store's arrays are "rung1" and "rung2"
            nulls[r] = stored[f"rung{r}"]
    print(f"loaded {STORE}")
for r in RUNGS:
    print(f"rung {r}: {nulls[r].size} null windows")

## 2. One null window, by hand (App. A.2)

A single star (rung 1's null), reduced as the ladder does: one source, then two from the one-source fit, and
$D_1$ the drop in $\chi^2$ over $\sigma_n^2$. The fits are nested, so $D_1 \ge 0$; most single stars give a small
$D_1$, and the margin is set so that 1 % exceed it.

In [ ]:
# one single star reduced by hand: fit one source, then two started from it; D_1 is the drop in chi^2 over
# sigma_n^2, compared with the margin Delta_1. fit_one[:2] keeps (positions, fluxes) and omits the residual sum
# of squares, which is fit_one[2]; fit_one[0][0] is the position of the one source; ins.TOY_MARGINS[0] is Delta_1
# step 1, the window. The generator of its noise: the seed is 0; the reason for that value: not stated
rng_demo = np.random.default_rng(0)
sigma_demo = sim.PRIMARY_FLUX / 100.0      # S/N 100: the noise sigma per sample
# sim.simulate_window: the scene through the LSF on the 18 samples, plus Gaussian noise of sigma sigma_demo per
# sample. The scene: one star at 37 mas with the primary's flux
window_demo = sim.simulate_window([37.0], [sim.PRIMARY_FLUX], inst, rng_demo, sigma_demo)
# step 2, the two fits
# red.fit_k: the best fit with K sources -> (positions, fluxes, residual sum of squares)
fit_one = red.fit_k(window_demo, 1, inst=inst)
fit_two = red.fit_k(window_demo, 2, previous=fit_one[:2], inst=inst)   # two sources, started from the one-source fit
# step 3, the statistic and the ladder's decision
chi2_one_source = fit_one[2]                 # the residual sum of squares of the one-source fit
chi2_two_sources = fit_two[2]                # the residual sum of squares of the two-source fit
D1 = (chi2_one_source - chi2_two_sources) / sigma_demo ** 2
position_one_source = fit_one[0][0]          # the fitted position of the one source [mas]
delta_1 = ins.TOY_MARGINS[0]               # the margin Delta_1
if D1 > delta_1:
    decision = "a second source"
else:
    decision = "one source"
print(f"one star at 37 mas: one source at {position_one_source:.1f} mas; D_1 = {D1:.2f} "
      f"against Delta_1 = {delta_1:.2f}"
      f" -> {decision}")
assert D1 >= 0.0                           # the fits are nested: the second source cannot make the fit worse

## 3. The stored null is the package's

The first 200 windows of each stream, recomputed now, must equal the stored
ones bit for bit.

In [ ]:
# recompute the first 200 null windows of each rung and require them to equal the store bit for bit
N_CHECK = 200
for r in RUNGS:                            # r: the rung's number
    rng_of_rung = streams()[r - 1]         # the rung's generator, at its start (index r - 1)
    fresh = sim.rung_null_statistic(rng_of_rung, r, N_CHECK, snr=NULL_SNR, inst=inst)
    # [:N_CHECK] keeps the first N_CHECK windows of the store
    assert np.array_equal(fresh, nulls[r][:N_CHECK]), \
        f"rung {r}: the package no longer reproduces the stored null; set FULL = True"
print(f"first {N_CHECK} windows of each rung recomputed: identical")

## 4. The margins (App. A.2)

$\Delta_K$ is the 99th percentile of each rung's null, equal to the frozen margins the package applies; a bootstrap
gives its statistical error. A χ² table does not apply (App. A.2): the added flux is non-negative and its position is
searched, so the null is not χ² with a fixed number of degrees of freedom. The cell shows how often the rung-1 null
exceeds the 99th percentiles of χ² with one and two degrees of freedom.

In [ ]:
# the margins: the sim.MARGIN_PERCENTILE (99th) percentile of each rung's null, which must equal ins.TOY_MARGINS
margins_list = []
for r in RUNGS:
    margins_list.append(np.percentile(nulls[r], sim.MARGIN_PERCENTILE))
margins = np.array(margins_list)           # (Delta_1, Delta_2)
assert np.array_equal(margins, ins.TOY_MARGINS)                    # equal to the frozen margins, to the last bit
assert np.array_equal(margins, inst.margins)                       # and to the margins that the instrument carries
assert np.array_equal(np.round(margins, 1), [11.1, 10.8])          # Appendix A

In [ ]:
# the statistical error of each margin by a bootstrap (the null resampled with replacement, 200 times), and the
# null's median; (null > margin).mean() is the fraction of the windows above the margin
N_BOOTSTRAP = 200                          # the number of resamples of the bootstrap
# the generator of the bootstrap: one stream for the two rungs, rung 1 first. The seed is 1; the reason for that
# value: not stated
boot_rng = np.random.default_rng(1)
for r, margin in zip(RUNGS, margins):      # r: the rung's number; margin: its Delta_r
    null = nulls[r]                        # D_r of the rung's null windows
    # the margin of each resample: as many windows as the null has, drawn with replacement, and their percentile
    boot = []
    for _ in range(N_BOOTSTRAP):           # one pass per resample; the counter is not used
        resample = boot_rng.choice(null, null.size)
        boot.append(np.percentile(resample, sim.MARGIN_PERCENTILE))
    fraction_above = (null > margin).mean()
    print(f"Delta_{r} = {margin:.2f} +/- {np.std(boot):.2f}   ({null.size} windows, "
          f"{fraction_above:.1%} above, median {np.median(null):.2f})")
print("the frozen margins, recovered exactly")

In [ ]:
# the chi^2 table (App. A.2): the 99th percentile of chi^2 with one and two degrees of freedom, and how often
# the rung-1 null exceeds it
for df in (1, 2):                                                 # what a chi^2 table would have promised: 1 %
    chi2_level = chi2.ppf(0.99, df)        # the level that chi^2 with df degrees of freedom exceeds in 1 %
    fraction_above = np.mean(nulls[1] > chi2_level)
    print(f"chi^2 with {df} dof: 99th percentile {chi2_level:.2f}; the rung-1 null exceeds it in "
          f"{fraction_above:.1%} of windows")

## 5. The null distributions

Each rung's $D_K$ over its null windows (log scale), the margin as a vertical line.

In [ ]:
# the figure: each rung's null distribution, its margin as a vertical line
PAPER_STYLE = {"font.size": 10.0, "xtick.labelsize": 9.0, "ytick.labelsize": 9.0, "axes.linewidth": 0.5,
               "xtick.direction": "in", "ytick.direction": "in"}
with plt.rc_context(PAPER_STYLE):                                         # the paper's style, this plot only
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.4))
    for ax, r, margin in zip(axes, RUNGS, margins):    # ax: the rung's panel; r: its number; margin: its Delta_r
        null = nulls[r]                                # D_r of the rung's null windows
        # 60 bin edges from 0 to the null's 99.9th percentile, or to 20 when that is smaller
        ax.hist(null, bins=np.linspace(0, max(20, np.percentile(null, 99.9)), 60), color="0.7")
        ax.axvline(margin, color="k", lw=1)
        ax.set_yscale("log")
        ax.set_xlabel(rf"$D_{r}$ on the rung-{r} null")
        # ax.get_ylim() returns (lower, upper); [1] keeps the upper limit
        ax.text(margin, ax.get_ylim()[1] * 0.3, rf" $\Delta_{r}$ = {margin:.2f}", fontsize=8)
    plt.tight_layout()
    plt.show()